# Comparaison Finale des 3 Modèles (EM/F1)

Notebook dédié pour comparer rapidement **Model v1**, **Model v2 amélioré** et **Model v3**.


## 1) Imports et Device (Apple MPS)


In [1]:
import os
import re
import json
import string
from collections import Counter

import torch
import torch.nn as nn

from models import QAModelV1, QAModelV2, QAModelV3

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


Device: mps


## 2) Configuration


In [2]:
data_path = 'data/'
train_path = os.path.join(data_path, 'train-v2.0.json')
dev_path = os.path.join(data_path, 'dev-v2.0.json')

COMPARISON_MAX_EXAMPLES = 2000

# Config v1/v2
V12_MAX_CONTEXT_LEN = 200
V12_MAX_QUESTION_LEN = 30
V12_EMBED_DIM = 100
V12_LSTM_UNITS = 128

# Config v3
V3_MAX_CONTEXT_LEN = 220
V3_MAX_QUESTION_LEN = 40
V3_EMBED_DIM = 200
V3_HIDDEN_SIZE = 128
V3_MIN_FREQ = 2
NO_ANSWER_THRESHOLD = 0.5
MAX_ANSWER_LEN = 30

ckpt_v1 = 'bidaf_model.pt'
ckpt_v2 = 'bidaf_model_improved.pt'
ckpt_v3 = 'bidaf_model_v3_best.pt'


## 3) Métriques Partagées


In [3]:
def _normalize_answer_cmp(s):
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())


def _em_cmp(pred, truth):
    return int(_normalize_answer_cmp(pred) == _normalize_answer_cmp(truth))


def _f1_cmp(pred, truth):
    p = _normalize_answer_cmp(pred).split()
    t = _normalize_answer_cmp(truth).split()
    if len(p) == 0 and len(t) == 0:
        return 1.0
    if len(p) == 0 or len(t) == 0:
        return 0.0
    common = Counter(p) & Counter(t)
    n = sum(common.values())
    if n == 0:
        return 0.0
    precision = n / len(p)
    recall = n / len(t)
    return 2 * precision * recall / (precision + recall)


## 4) Pipeline d'évaluation Model v1 / v2


In [4]:
def tokenize_basic(text):
    return text.lower().split()


def load_squad_basic(path, max_examples=None):
    with open(path, 'r', encoding='utf-8') as f:
        squad = json.load(f)

    examples = []
    for article in squad['data']:
        for paragraph in article['paragraphs']:
            context = paragraph['context']
            context_tokens = tokenize_basic(context)
            for qa in paragraph['qas']:
                answer_obj = None
                if len(qa.get('answers', [])) > 0:
                    answer_obj = qa['answers'][0]
                elif len(qa.get('plausible_answers', [])) > 0:
                    answer_obj = qa['plausible_answers'][0]
                if answer_obj is None:
                    continue

                question_tokens = tokenize_basic(qa['question'])

                char_count = 0
                start_token = end_token = None
                for i, token in enumerate(context_tokens):
                    if char_count <= answer_obj['answer_start'] < char_count + len(token) + 1:
                        start_token = i
                    if char_count <= answer_obj['answer_start'] + len(answer_obj['text']) <= char_count + len(token) + 1:
                        end_token = i
                        break
                    char_count += len(token) + 1

                if start_token is not None and end_token is not None:
                    examples.append({
                        'context_tokens': context_tokens,
                        'question_tokens': question_tokens,
                        'start': start_token,
                        'end': end_token,
                        'raw_answer': answer_obj['text']
                    })

                if max_examples is not None and len(examples) >= max_examples:
                    return examples

    return examples


def build_vocab_basic(examples, min_freq=2):
    counter = Counter()
    for ex in examples:
        counter.update(ex['context_tokens'])
        counter.update(ex['question_tokens'])
    vocab_obj = {'<PAD>': 0, '<UNK>': 1}
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab_obj[token] = len(vocab_obj)
    return vocab_obj


def encode_basic(tokens, vocab_obj, max_len):
    ids = [vocab_obj.get(t, vocab_obj['<UNK>']) for t in tokens][:max_len]
    return ids + [vocab_obj['<PAD>']] * (max_len - len(ids))


@torch.no_grad()
def predict_span_basic(model_obj, context_tokens, question_tokens, vocab_obj):
    model_obj.eval()
    ctx = torch.LongTensor([encode_basic(context_tokens, vocab_obj, V12_MAX_CONTEXT_LEN)]).to(device)
    qst = torch.LongTensor([encode_basic(question_tokens, vocab_obj, V12_MAX_QUESTION_LEN)]).to(device)
    s_logits, e_logits = model_obj(ctx, qst)
    s_idx = torch.argmax(s_logits[0]).item()
    e_idx = torch.argmax(e_logits[0]).item()
    if e_idx < s_idx:
        e_idx = s_idx
    s_idx = min(s_idx, len(context_tokens) - 1)
    e_idx = min(e_idx, len(context_tokens) - 1)
    return ' '.join(context_tokens[s_idx:e_idx+1])


def eval_basic_model(model_obj, examples, vocab_obj, max_examples=2000):
    data = examples[:max_examples] if max_examples is not None else examples
    em_total = 0
    f1_total = 0.0
    n = 0
    for ex in data:
        pred = predict_span_basic(model_obj, ex['context_tokens'], ex['question_tokens'], vocab_obj)
        truth = ex['raw_answer']
        em_total += _em_cmp(pred, truth)
        f1_total += _f1_cmp(pred, truth)
        n += 1
    return {
        'n_eval': n,
        'em': 100.0 * em_total / max(n, 1),
        'f1': 100.0 * f1_total / max(n, 1),
    }


## 5) Pipeline d'évaluation Model v3


In [5]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]")


def tokenize_advanced(text):
    return TOKEN_PATTERN.findall(text.lower())


def char_to_token_span(context, context_tokens, answer_start, answer_text):
    lowered = context.lower()
    offsets = []
    cursor = 0
    for tok in context_tokens:
        idx = lowered.find(tok, cursor)
        if idx == -1:
            idx = lowered.find(tok)
            if idx == -1:
                return None, None
        offsets.append((idx, idx + len(tok)))
        cursor = idx + len(tok)

    ans_start = answer_start
    ans_end = answer_start + len(answer_text)
    s_tok = e_tok = None
    for i, (s, e) in enumerate(offsets):
        if s <= ans_start < e:
            s_tok = i
        if s < ans_end <= e:
            e_tok = i
            break
    return s_tok, e_tok


def load_squad_v3(path, max_examples=None):
    with open(path, 'r', encoding='utf-8') as f:
        squad = json.load(f)

    examples = []
    for article in squad['data']:
        for paragraph in article['paragraphs']:
            context = paragraph['context']
            context_tokens = tokenize_advanced(context)

            for qa in paragraph['qas']:
                question_tokens = tokenize_advanced(qa['question'])
                is_impossible = qa.get('is_impossible', False)

                if is_impossible:
                    examples.append({
                        'context_tokens': context_tokens,
                        'question_tokens': question_tokens,
                        'start': 0,
                        'end': 0,
                        'has_answer': 0,
                        'raw_answer': ''
                    })
                else:
                    answers = qa.get('answers', [])
                    if len(answers) == 0:
                        continue

                    ans = answers[0]
                    s_tok, e_tok = char_to_token_span(context, context_tokens, ans['answer_start'], ans['text'].lower())
                    if s_tok is None or e_tok is None:
                        continue

                    examples.append({
                        'context_tokens': context_tokens,
                        'question_tokens': question_tokens,
                        'start': s_tok + 1,
                        'end': e_tok + 1,
                        'has_answer': 1,
                        'raw_answer': ans['text']
                    })

                if max_examples is not None and len(examples) >= max_examples:
                    return examples

    return examples


def build_vocab_v3(examples, min_freq=2):
    counter = Counter()
    for ex in examples:
        counter.update(ex['context_tokens'])
        counter.update(ex['question_tokens'])

    vocab_obj = {'<PAD>': 0, '<UNK>': 1, '<CLS>': 2}
    for token, freq in counter.items():
        if freq >= min_freq and token not in vocab_obj:
            vocab_obj[token] = len(vocab_obj)
    return vocab_obj


def encode_v3(tokens, vocab_obj, max_len, add_cls=False):
    if add_cls:
        tokens = ['<CLS>'] + tokens
    ids = [vocab_obj.get(t, vocab_obj['<UNK>']) for t in tokens][:max_len]
    return ids + [vocab_obj['<PAD>']] * (max_len - len(ids))


def clean_prediction(prediction):
    prediction = re.sub(r"^[\s\(\)\[\],;:\.""]+", "", prediction)
    prediction = re.sub(r"[\s\(\)\[\],;:\.""]+$", "", prediction)
    prediction = re.sub(r'\s+', ' ', prediction).strip()
    return prediction


@torch.no_grad()
def decode_best_span_v3(start_logits, end_logits, max_answer_len=30):
    best_score = -1e30
    best_s, best_e = 0, 0
    L = start_logits.size(0)

    for s in range(L):
        e_max = min(L - 1, s + max_answer_len - 1)
        e_rel = torch.argmax(end_logits[s:e_max+1]).item()
        e = s + e_rel
        score = start_logits[s].item() + end_logits[e].item()
        if score > best_score:
            best_score = score
            best_s, best_e = s, e
    return best_s, best_e


@torch.no_grad()
def predict_answer_v3(model_obj, context_tokens, question_tokens, vocab_obj, threshold=0.5):
    model_obj.eval()

    ctx_ids = torch.LongTensor([encode_v3(context_tokens, vocab_obj, V3_MAX_CONTEXT_LEN, add_cls=True)]).to(device)
    qst_ids = torch.LongTensor([encode_v3(question_tokens, vocab_obj, V3_MAX_QUESTION_LEN, add_cls=False)]).to(device)

    s_logits, e_logits, na_logit = model_obj(ctx_ids, qst_ids)
    s_logits = s_logits[0]
    e_logits = e_logits[0]

    p_no_answer = torch.sigmoid(na_logit[0]).item()
    if p_no_answer >= threshold:
        return '', 0, 0, p_no_answer

    s_idx, e_idx = decode_best_span_v3(s_logits, e_logits, MAX_ANSWER_LEN)

    if s_idx == 0 or e_idx == 0:
        return '', s_idx, e_idx, p_no_answer

    s_txt = s_idx - 1
    e_txt = min(e_idx - 1, len(context_tokens) - 1)
    if e_txt < s_txt:
        return '', s_idx, e_idx, p_no_answer

    pred = ' '.join(context_tokens[s_txt:e_txt+1])
    return clean_prediction(pred), s_idx, e_idx, p_no_answer


def eval_v3_model(model_obj, examples, vocab_obj, max_examples=2000):
    data = examples[:max_examples] if max_examples is not None else examples
    em_total = 0
    f1_total = 0.0
    n = 0

    for ex in data:
        if ex['start'] >= V3_MAX_CONTEXT_LEN or ex['end'] >= V3_MAX_CONTEXT_LEN:
            continue
        pred, _, _, _ = predict_answer_v3(model_obj, ex['context_tokens'], ex['question_tokens'], vocab_obj, NO_ANSWER_THRESHOLD)
        truth = ex['raw_answer']
        em_total += _em_cmp(pred, truth)
        f1_total += _f1_cmp(pred, truth)
        n += 1

    return {
        'n_eval': n,
        'em': 100.0 * em_total / max(n, 1),
        'f1': 100.0 * f1_total / max(n, 1),
    }


## 6) Chargement des Données + Checkpoints


In [6]:
results = []

if os.path.exists(ckpt_v1) and os.path.exists(ckpt_v2):
    print('Chargement et évaluation des modèles v1/v2...')
    train_basic = load_squad_basic(train_path)
    dev_basic = load_squad_basic(dev_path, max_examples=COMPARISON_MAX_EXAMPLES)
    vocab_basic = build_vocab_basic(train_basic, min_freq=2)

    v1 = QAModelV1(len(vocab_basic), V12_EMBED_DIM, V12_LSTM_UNITS, V12_MAX_CONTEXT_LEN).to(device)
    v1.load_state_dict(torch.load(ckpt_v1, map_location=device))
    m1 = eval_basic_model(v1, dev_basic, vocab_basic, COMPARISON_MAX_EXAMPLES)
    results.append({'model': 'Model v1', **m1})

    v2 = QAModelV2(len(vocab_basic), V12_EMBED_DIM, V12_LSTM_UNITS, V12_MAX_CONTEXT_LEN).to(device)
    v2.load_state_dict(torch.load(ckpt_v2, map_location=device))
    m2 = eval_basic_model(v2, dev_basic, vocab_basic, COMPARISON_MAX_EXAMPLES)
    results.append({'model': 'Model v2 (amélioré)', **m2})
else:
    print('Checkpoints v1/v2 introuvables, comparaison partielle.')

if os.path.exists(ckpt_v3):
    print('Chargement et évaluation du modèle v3...')
    train_v3 = load_squad_v3(train_path)
    dev_v3 = load_squad_v3(dev_path, max_examples=COMPARISON_MAX_EXAMPLES)
    vocab_v3 = build_vocab_v3(train_v3, min_freq=V3_MIN_FREQ)

    model_v3 = QAModelV3(len(vocab_v3), V3_EMBED_DIM, V3_HIDDEN_SIZE).to(device)
    model_v3.load_state_dict(torch.load(ckpt_v3, map_location=device))
    m3 = eval_v3_model(model_v3, dev_v3, vocab_v3, COMPARISON_MAX_EXAMPLES)
    results.append({'model': 'Model v3', **m3})
else:
    print('Checkpoint v3 introuvable:', ckpt_v3)


Chargement et évaluation des modèles v1/v2...


/Users/albanecoiffe/Downloads/NLP/.venv/lib/python3.14/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Chargement et évaluation du modèle v3...


## 7) Tableau de Comparaison Finale


In [7]:
if len(results) == 0:
    print('Aucun résultat à afficher.')
else:
    print('| Modèle | N évalués | EM (%) | F1 (%) |')
    print('|---|---:|---:|---:|')
    for r in results:
        print(f"| {r['model']} | {r['n_eval']} | {r['em']:.2f} | {r['f1']:.2f} |")

    best = sorted(results, key=lambda x: x['f1'], reverse=True)[0]
    print(f"Meilleur modèle (F1): {best['model']} ({best['f1']:.2f}%)")


| Modèle | N évalués | EM (%) | F1 (%) |
|---|---:|---:|---:|
| Model v1 | 2000 | 2.40 | 7.97 |
| Model v2 (amélioré) | 2000 | 10.55 | 18.73 |
| Model v3 | 1998 | 37.09 | 39.16 |
Meilleur modèle (F1): Model v3 (39.16%)
